# Executive Order Analysis

Analysis over `data/analysis.db`: 1,534 Executive Orders, EO 12890 (1993-12-30) to
EO 14423 (2026-08-28), extracted by `openai/gpt-oss-120b` under prompt v7 (run 11).

Before quoting any number from this notebook, read
[INVESTIGATORS_CHEAT_SHEET.md](../INVESTIGATORS_CHEAT_SHEET.md). The four things it
says most often:

- **Recall was never measured.** "The model extracted X from 66% of orders" is
  supportable; "34% of orders have no X" is not.
- **Count agencies through `agency_taskings`**, ranked by distinct orders, excluding
  `kind IN ('collective', 'generic')`.
- **Filter relationships to `authoritative = 1`** (the `revocation_network` view
  already does).
- **Only `source_quote` is verified.** `summary` and `task` are model prose.

Every cell runs from the repository root with the project venv:

```sh
pip install -e ".[analysis]"
PYTHONPATH=src .venv/bin/jupyter lab notebooks/analysis.ipynb
```

## Setup

In [1]:
import re
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from eo.grounding import is_grounded  # noqa: E402

DB = ROOT / "data" / "analysis.db"
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)


def q(sql: str, **params) -> pd.DataFrame:
    """Run a query against analysis.db (read-only) and return a DataFrame."""
    return pd.read_sql_query(sql, con, params=params or None)


pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 140)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (9, 4.5)

# Presidents in chronological order, for consistent axes and tables.
PRESIDENTS = q("SELECT president FROM orders GROUP BY 1 ORDER BY MIN(signing_date)")["president"].tolist()
SHORT = {"William J. Clinton": "Clinton", "George W. Bush": "G.W. Bush", "Barack Obama": "Obama",
         "Donald Trump": "Trump", "Joseph R. Biden Jr.": "Biden"}
print(DB, "opened read-only")

Matplotlib is building the font cache; this may take a moment.


/Users/davidkolet-tassara/Documents/EO_Analysis/data/analysis.db opened read-only


## Sanity checks

The three checks the cheat sheet asks for before anything is published: which model
produced the data, that every stored quote really is in its order's text, and what is
already known to be imperfect.

In [2]:
q("SELECT run_id, model, prompt_version, ROUND(cost_usd, 2) AS cost_usd, coverage FROM run_metadata").T

,0
run_id,11
model,openai/gpt-oss-120b
prompt_version,v7
cost_usd,1.05
coverage,"Executive Orders with full text in the Federal Register API, which begins ~1..."


In [3]:
rows = con.execute(
    "SELECT c.source_quote, t.body_text FROM all_claims c JOIN order_text t USING (document_number)"
).fetchall()
grounded = sum(is_grounded(quote, body) for quote, body in rows)
print(f"{grounded} / {len(rows)} stored quotes appear in their order's text")
assert grounded == len(rows), "a stored quote is not in its source: the build is broken"

8989 / 8989 stored quotes appear in their order's text


In [4]:
q("SELECT kind, COUNT(*) AS items FROM review_queue GROUP BY 1 ORDER BY 2 DESC")

,kind,items
0,dropped_unverifiable,259
1,ungrounded_agencies_tasked,241
2,ungrounded_deadlines,156
3,ungrounded_authorities,77
4,ungrounded_relationships,66
5,relationship,63
6,other_without_reason,4


## Shape of the corpus

Orders per president, per *active* year. Trump's two terms are non-contiguous, so a
first-to-last date span would attribute 2021-24 to him and understate his rate.

In [5]:
shape = q("""
SELECT president,
       COUNT(*) AS orders,
       MIN(signing_date) AS first, MAX(signing_date) AS last,
       COUNT(DISTINCT substr(signing_date, 1, 4)) AS active_years,
       ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT substr(signing_date, 1, 4)), 1) AS per_active_year
FROM orders GROUP BY president ORDER BY MIN(signing_date)
""")
shape

,president,orders,first,last,active_years,per_active_year
0,William J. Clinton,308,1993-12-30,2001-01-18,9,34.2
1,George W. Bush,291,2001-01-29,2009-01-16,9,32.3
2,Barack Obama,276,2009-01-21,2017-01-17,9,30.7
3,Donald Trump,497,2017-01-20,2026-08-28,7,71.0
4,Joseph R. Biden Jr.,162,2021-01-20,2025-01-19,5,32.4


In [6]:
q("""
SELECT 'orders' AS t, COUNT(*) AS n FROM orders
UNION ALL SELECT 'agencies_tasked', COUNT(*) FROM agencies_tasked
UNION ALL SELECT 'deadlines', COUNT(*) FROM deadlines
UNION ALL SELECT 'authorities', COUNT(*) FROM authorities
UNION ALL SELECT 'relationships (authoritative)', COUNT(*) FROM relationships WHERE authoritative = 1
UNION ALL SELECT 'agencies (canonical)', COUNT(*) FROM agencies
""").set_index("t")

,n
t,
orders,1534
agencies_tasked,3195
deadlines,2240
authorities,1707
relationships (authoritative),4848
agencies (canonical),598


## Analysis

Sections below are added as questions are asked. Each one states the query it rests on
and the caveat that applies to it.